# 06 — Analyse d'erreurs et export XML-TEI

1. Analyse des erreurs de chaque modèle à partir de leurs prédictions sur le test (fichiers CSV produits par les notebooks 03 à 05).
2. Export des transcriptions de Kraken affiné en XML-TEI, un fichier par manuscrit de test, validé contre `tei_all`.

**Environnement** : Google Colab, sans GPU.

In [ ]:
!pip install -q rapidfuzz jiwer lxml

import re, csv, json, unicodedata, jiwer
import xml.etree.ElementTree as ET
from pathlib import Path
from collections import Counter, defaultdict
from rapidfuzz.distance import Levenshtein
from PIL import Image

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/projet_vlm")
except ImportError:
    ROOT = Path("projet_vlm")

FICHIERS = {
    "Kraken tel quel": "predictions_kraken_poly.csv",
    "Kraken affiné":   "predictions_kraken_ft.csv",
    "Qwen tel quel":   "predictions_qwen_zeroshot.csv",
    "Qwen + LoRA":     "predictions_qwen_lora.csv",
    "Qwen + LoRA ×2":  "predictions_qwen_lora_x2.csv",
    "Qwen + LoRA vision": "predictions_qwen_lora_vision.csv",
}
FICHIERS = {nom: ROOT / f for nom, f in FICHIERS.items() if (ROOT / f).exists()}

def nfd(s):
    return unicodedata.normalize("NFD", s or "").strip()

def charger(chemin):
    with open(chemin, encoding="utf-8") as f:
        lignes = list(csv.DictReader(f))
    return [nfd(r["vrai"]) for r in lignes], [nfd(r["pred"]) for r in lignes]

## 1. Types d'erreurs

Alignement de Levenshtein entre la vérité et la prédiction. Chaque opération est une **substitution** (lettre mal lue), une **suppression** (lettre oubliée) ou une **insertion** (lettre ajoutée : signature des hallucinations). Les taux sont rapportés au nombre de caractères de référence. Le **CER sans espaces** isole la part des erreurs due à la coupure des mots.

In [ ]:
def analyser(vrais, preds):
    n = sum(len(v) for v in vrais)
    ops, confusions = Counter(), Counter()
    for v, p in zip(vrais, preds):
        for op in Levenshtein.editops(v, p):
            ops[op.tag] += 1
            if op.tag == "replace":
                confusions[(v[op.src_pos], p[op.dest_pos])] += 1
    stats = {
        "CER": jiwer.cer(vrais, preds),
        "CER sans espaces": jiwer.cer([v.replace(" ", "") for v in vrais], [p.replace(" ", "") for p in preds]),
        "substitutions": ops["replace"] / n,
        "suppressions": ops["delete"] / n,
        "insertions": ops["insert"] / n,
    }
    return stats, confusions

resultats = {nom: analyser(*charger(c)) for nom, c in FICHIERS.items()}

print(f"{'':16}" + "".join(f"{k:>18}" for k in next(iter(resultats.values()))[0]))
for nom, (stats, _) in resultats.items():
    print(f"{nom:16}" + "".join(f"{v:>17.1%} " for v in stats.values()))

## 2. Caractères bien lus par catégorie, et confusions les plus fréquentes

In [ ]:
def categorie(c):
    if c == " ":
        return "espaces"
    if c.isascii() and c.isalpha():
        return "lettres a-z"
    if c.isascii():
        return "ponctuation / chiffres"
    return "abréviations et signes spéciaux"

def rappel_par_categorie(vrais, preds):
    total, rates = Counter(), Counter()
    for v, p in zip(vrais, preds):
        touches = {op.src_pos for op in Levenshtein.editops(v, p) if op.tag in ("replace", "delete")}
        for i, c in enumerate(v):
            total[categorie(c)] += 1
            rates[categorie(c)] += i in touches
    return {k: 1 - rates[k] / total[k] for k in total}, total

for nom, chemin in FICHIERS.items():
    rappel, total = rappel_par_categorie(*charger(chemin))
    print(f"\n=== {nom} ===")
    for k in sorted(total, key=lambda k: -total[k]):
        print(f"  {k:32} {total[k]:6} caractères   bien lus : {rappel[k]:.1%}")
    print("  Confusions les plus fréquentes (vrai → prédit) :")
    for (a, b), n in resultats[nom][1].most_common(10):
        print(f"    {a!r:>6} → {b!r:<6} {n:4}×")

json.dump({nom: {k: round(v, 4) for k, v in s.items()} for nom, (s, _) in resultats.items()},
          open(ROOT / "analyse_erreurs.json", "w"), ensure_ascii=False, indent=1)

## 3. Kraken affiné et Qwen adapté sont-ils vraiment différents ?

Bootstrap apparié : on tire 10 000 fois, avec remise, 574 lignes du test, et on recalcule à chaque fois l'écart de CER entre les deux modèles **sur les mêmes lignes**. Si l'intervalle à 95 % contient 0, l'écart n'est pas significatif. Limite : les lignes d'un même manuscrit ne sont pas indépendantes, l'intervalle est donc probablement trop étroit.

In [ ]:
import random

def bootstrap(nom_a, nom_b, n_tirages=10_000, graine=0):
    with open(FICHIERS[nom_a], encoding="utf-8") as f:
        lignes = list(csv.DictReader(f))
    vrais, pa = charger(FICHIERS[nom_a])
    _, pb = charger(FICHIERS[nom_b])
    ea = [Levenshtein.distance(v, p) for v, p in zip(vrais, pa)]
    eb = [Levenshtein.distance(v, p) for v, p in zip(vrais, pb)]
    nc = [len(v) for v in vrais]
    rng = random.Random(graine)

    def intervalle(indices):
        ecarts = []
        for _ in range(n_tirages):
            tirage = [rng.choice(indices) for _ in indices]
            ecarts.append((sum(eb[i] for i in tirage) - sum(ea[i] for i in tirage)) / sum(nc[i] for i in tirage))
        ecarts.sort()
        reel = (sum(eb[i] for i in indices) - sum(ea[i] for i in indices)) / sum(nc[i] for i in indices)
        return reel, ecarts[int(0.025 * n_tirages)], ecarts[int(0.975 * n_tirages)]

    print(f"Écart de CER ({nom_b}) − ({nom_a}), intervalle à 95 % :")
    reel, bas, haut = intervalle(list(range(len(vrais))))
    print(f"  {'global':25} {reel:+.1%}  [{bas:+.1%} ; {haut:+.1%}]")
    for m in sorted({r["manuscrit"] for r in lignes}):
        reel, bas, haut = intervalle([i for i, r in enumerate(lignes) if r["manuscrit"] == m])
        print(f"  {m:25} {reel:+.1%}  [{bas:+.1%} ; {haut:+.1%}]")

if "Qwen + LoRA vision" in FICHIERS:
    bootstrap("Kraken affiné", "Qwen + LoRA vision")


## 4. Export XML-TEI

Transcriptions de **Kraken affiné** (meilleur modèle), un fichier par manuscrit de test :
- `<teiHeader>` : source (corpus HTRogène, licence CC-BY), modèle et CER mesuré sur le manuscrit ; aucune cote inventée (identifiant HTRogène dans `idno`) ;
- `<facsimile>` : une `<surface>` par page, une `<zone>` par ligne avec son polygone ALTO ;
- `<body>` : transcription diplomatique, un `<lb facs="#…"/>` par ligne ;
- zones Segmonto converties lorsqu'elles sont annotées.

Texte en NFC (le NFD ne sert qu'au calcul du CER) ; identifiants `xml:id` en ASCII.

In [ ]:
!git clone -q --depth 1 https://github.com/HTRogene/occitan

ALTO = {"a": "http://www.loc.gov/standards/alto/ns-v4#"}
TEI = "http://www.tei-c.org/ns/1.0"
XML_ID = "{http://www.w3.org/XML/1998/namespace}id"
ET.register_namespace("", TEI)
SRC = Path("occitan/data")

def t(tag):
    return f"{{{TEI}}}{tag}"

def nfc(s):
    return unicodedata.normalize("NFC", s or "")

def ident(s):
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return "p_" + re.sub(r"[^A-Za-z0-9_.-]", "_", s)

with open(ROOT / "predictions_kraken_ft.csv", encoding="utf-8") as f:
    PRED = {nfc(Path(r["image"]).name): r for r in csv.DictReader(f)}

CONVERSION = {
    "NumberingZone":    ("fw", {"type": "folNum"}),
    "RunningTitleZone": ("fw", {"type": "header"}),
    "MarginTextZone":   ("note", {"place": "margin"}),
}

In [ ]:
def tei_manuscrit(manuscrit):
    lignes_ms = [r for r in PRED.values() if r["manuscrit"] == manuscrit]
    cer = jiwer.cer([r["vrai"] for r in lignes_ms], [r["pred"] for r in lignes_ms])

    racine = ET.Element(t("TEI"))
    header = ET.SubElement(racine, t("teiHeader"))
    fd = ET.SubElement(header, t("fileDesc"))
    ts = ET.SubElement(fd, t("titleStmt"))
    ET.SubElement(ts, t("title")).text = f"Transcription automatique — {manuscrit}"
    resp = ET.SubElement(ts, t("respStmt"))
    ET.SubElement(resp, t("resp")).text = "Transcription automatique et encodage"
    ET.SubElement(resp, t("name")).text = "Ezéchiel Sawadogo"
    ET.SubElement(ET.SubElement(fd, t("publicationStmt")), t("p")).text = "Document de travail, non publié."
    sd = ET.SubElement(fd, t("sourceDesc"))
    ms = ET.SubElement(ET.SubElement(sd, t("msDesc")), t("msIdentifier"))
    ET.SubElement(ms, t("idno"), type="htrogene").text = manuscrit
    ET.SubElement(sd, t("bibl")).text = (
        "Images et segmentation : HTRogène, Occitan corpus of ground-truth for HTR "
        "(Wiedner, Couffignal, Chagué, Clérice), CC-BY 4.0, https://github.com/HTRogene/occitan")
    ed = ET.SubElement(header, t("encodingDesc"))
    app = ET.SubElement(ET.SubElement(ed, t("appInfo")), t("application"), ident="kraken", version="7.1.1")
    ET.SubElement(app, t("label")).text = "Kraken"
    ET.SubElement(app, t("desc")).text = (
        "Modèle CATMuS Medieval 1.5.0 affiné sur les manuscrits d'entraînement HTRogène. "
        f"Manuscrit jamais vu à l'entraînement. CER mesuré sur ce manuscrit : {cer:.1%}.")
    ET.SubElement(ET.SubElement(ed, t("editorialDecl")), t("p")).text = (
        "Transcription diplomatique automatique, non corrigée. Abréviations non développées. "
        "Une balise lb par ligne du manuscrit, reliée à sa zone dans le facsimilé. "
        "Zones Segmonto converties : NumberingZone en fw[@type='folNum'], RunningTitleZone en "
        "fw[@type='header'], MarginTextZone en note[@place='margin'], HeadingLine en seg[@type='heading']."
        " Le typage des zones dépend de l'annotation source, hétérogène selon les manuscrits "
        "(étiquettes Segmonto absentes pour certains) : le paratexte (ex-libris, titres) "
        "n'est donc pas toujours distingué du texte.")

    facs = ET.SubElement(racine, t("facsimile"))
    ab = ET.SubElement(ET.SubElement(ET.SubElement(ET.SubElement(racine, t("text")), t("body")),
                                     t("div"), type="transcription"), t("ab"))

    for xml_path in sorted((SRC / manuscrit).glob("*.chocomufin.xml")):
        page_nom = xml_path.name.replace(".chocomufin.xml", "")
        img_path = xml_path.parent / f"{page_nom}.jpg"
        if not img_path.exists():
            continue
        larg, haut = Image.open(img_path).size
        pid = ident(page_nom)
        surface = ET.SubElement(facs, t("surface"), {XML_ID: pid, "ulx": "0", "uly": "0",
                                                     "lrx": str(larg), "lry": str(haut)})
        ET.SubElement(surface, t("graphic"), url=img_path.name)
        ET.SubElement(ab, t("pb"), facs=f"#{pid}")

        doc = ET.parse(xml_path).getroot()
        TAGS = {tag.get("ID"): tag.get("LABEL") for tag in doc.findall(".//a:OtherTag", ALTO)}
        i = -1   # même numérotation des lignes que dans le notebook 02
        for bloc in doc.findall(".//a:TextBlock", ALTO):
            zone_type = TAGS.get(bloc.get("TAGREFS", ""), "MainZone")
            parent = ET.SubElement(ab, t(CONVERSION[zone_type][0]), CONVERSION[zone_type][1]) \
                     if zone_type in CONVERSION else ab
            for ligne in bloc.findall("a:TextLine", ALTO):
                i += 1
                nom = nfc(f"{manuscrit}__{page_nom}__{i:03d}.png")
                if nom not in PRED:
                    continue
                c = ligne.find("a:Shape/a:Polygon", ALTO).get("POINTS").split()
                zid = f"{pid}_l{i:03d}"
                ET.SubElement(surface, t("zone"), {XML_ID: zid, "type": zone_type,
                              "points": " ".join(f"{c[k]},{c[k+1]}" for k in range(0, len(c), 2))})
                ET.SubElement(parent, t("lb"), facs=f"#{zid}")
                texte = nfc(PRED[nom]["pred"])
                if TAGS.get(ligne.get("TAGREFS", "")) == "HeadingLine":
                    ET.SubElement(parent, t("seg"), type="heading").text = texte
                else:
                    parent[-1].tail = texte
            if parent is not ab and len(parent) == 0:
                ab.remove(parent)

    ET.indent(racine, space="  ")
    sortie = ROOT / "tei" / f"{manuscrit}.xml"
    sortie.parent.mkdir(exist_ok=True)
    ET.ElementTree(racine).write(sortie, encoding="utf-8", xml_declaration=True)
    return sortie, cer

## Génération et validation contre `tei_all`

In [ ]:
!wget -q -O tei_all.rng https://tei-c.org/release/xml/tei/custom/schema/relaxng/tei_all.rng
from lxml import etree
schema = etree.RelaxNG(etree.parse("tei_all.rng"))
ns = {"tei": TEI}

for m in ["paris-bnf-arsenal-6355", "additional-21218", "paris-bnf-naf-6195"]:
    sortie, cer = tei_manuscrit(m)
    doc = etree.parse(str(sortie))
    compte = {nom: len(doc.findall(chemin, ns)) for nom, chemin in [
        ("lb", ".//tei:lb"), ("fw", ".//tei:fw"), ("note", ".//tei:note"), ("seg", ".//tei:seg")]}
    print(f"{m:25} CER {cer:.1%}  valide : {schema.validate(doc)}  {compte}")
    for err in list(schema.error_log)[:3]:
        print("   ligne", err.line, ":", err.message)